# H05 — 05_hyperparameters/regularization / validación

Evaluación independiente con el evaluador oficial; barrido de confianza solo en validación interna. Guardar AP/F1 por clase, PR, confusión y diagnóstico de errores, tamaños y densidad.

Preparado para Colab T4. Entrenamientos, evaluaciones y benchmarks desactivados. Configurar las mismas rutas persistentes en todos los notebooks. Nunca consultar test durante selección.

In [ ]:
from pathlib import Path
import base64
import hashlib
import io
import subprocess
import sys
import zipfile

# This cell carries code/configuration only. No images, weights or results.
# Editing src/configs locally requires rebuilding this snapshot with
# scripts/package_colab_notebooks.ps1 before running a remote notebook.
_CODE_BUNDLE_SHA256 = '77c7317be58b365ee6bcbfe862f9f96d712e6188bf231bbc93d8861a3c3934cc'
_CODE_BUNDLE = '...'
REPO_ROOT = None  # Optional explicit checkout path; normally leave automatic.
INSTALL_MISSING_NOTEBOOK_DEPS = True
RUN_SETUP = False  # Full training dependencies; no experiments are launched.
RUN_MOUNT_DRIVE = False
DATASET_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vgxmmpi5zclSXRo_y2GwNIkx35QFqr5t'
DATASET_ROOT = '/content/spacelab_dataset'
STORAGE = '/content/drive/MyDrive/SpaceLAB_experiments/session_01'

if REPO_ROOT is not None:
    REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
elif sys.platform != 'win32' and Path('/content').is_dir():
    # Version-specific directory: never overwrite an older checkout or dataset.
    REPO_ROOT = Path('/content/SpaceLAB_runtime') / _CODE_BUNDLE_SHA256[:16]
else:
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p/'src/spacelab/notebook_setup.py').is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Abra el notebook dentro de SpaceLAB o conecte el kernel Colab.')

if sys.platform != 'win32' and Path('/content/SpaceLAB_runtime') == REPO_ROOT.parent:
    _payload = base64.b64decode(_CODE_BUNDLE)
    if hashlib.sha256(_payload).hexdigest() != _CODE_BUNDLE_SHA256:
        raise ValueError('El paquete de código está incompleto o alterado; recargue el notebook.')
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    _root = REPO_ROOT.resolve()
    with zipfile.ZipFile(io.BytesIO(_payload)) as _archive:
        for _entry in _archive.infolist():
            _target = (_root / _entry.filename).resolve()
            if _root not in _target.parents:
                raise ValueError('Ruta fuera del paquete de código: ' + _entry.filename)
            _contents = _archive.read(_entry)
            if _target.exists():
                if _target.read_bytes() != _contents:
                    raise RuntimeError('Código remoto editado en ' + str(_target)
                                       + '. Consérvelo y configure REPO_ROOT explícitamente.')
            else:
                _target.parent.mkdir(parents=True, exist_ok=True)
                _target.write_bytes(_contents)
    del _payload

if not (REPO_ROOT/'src/spacelab/notebook_setup.py').is_file():
    raise FileNotFoundError('REPO_ROOT no contiene el paquete actualizado: ' + str(REPO_ROOT))
print('Código del proyecto disponible en:', REPO_ROOT)
if RUN_MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if RUN_SETUP:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT/'requirements.txt')])
sys.path.insert(0, str(REPO_ROOT/'src'))
# Refuse mixing already-imported modules from a different runtime snapshot.
_loaded = sys.modules.get('spacelab')
if _loaded is not None and Path(_loaded.__file__).resolve().parent != (REPO_ROOT/'src/spacelab').resolve():
    raise RuntimeError('Otro SpaceLAB ya está importado. Reinicie el kernel para usar este código.')
from spacelab.notebook_setup import prepare_notebook
prepare_notebook(install_missing=INSTALL_MISSING_NOTEBOOK_DEPS)
from spacelab.time_budget import launch
from spacelab.splits import load_protocol
from spacelab.experiment_registry import config_from, collect, freeze_selection
from spacelab.io import read_json
from IPython.display import display, Markdown
PROTOCOL_PATH = Path(STORAGE)/'protocol.json'
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_BENCHMARK = False
RUN_SPLITS = False
ENABLE_OPTIONAL = False
SELECTED_RUN = None
RUN_PATH = None
VALIDATION_RUN = None
RESUME_CHECKPOINT = None

## Configuración editable y protocolo compartido

In [ ]:
CONFIG_FILE = REPO_ROOT/'experiments/05_hyperparameters/regularization/config.yaml'
# Cambiar a config_low.yaml/config_sgd.yaml/etc. donde exista.
protocol = load_protocol(PROTOCOL_PATH) if PROTOCOL_PATH.exists() else None
print('Protocolo:', protocol['protocol_id'] if protocol else 'Pendiente: ejecutar 00_create_splits')
print('Configuración:', CONFIG_FILE)


## Acción explícita, presupuesto y guardado de resultados

In [ ]:
if RUN_EVALUATION:
    if not RUN_PATH:
        raise ValueError('Indique RUN_PATH con el directorio de entrenamiento o fold que contiene checkpoints/best.pt')
    result = launch('validate', {'dataset': DATASET_ROOT, 'run': RUN_PATH}, STORAGE, cap_seconds=300)
    display(result)
    print('Figuras y métricas en el directorio de validación devuelto, sin sobrescribir otros jobs.')
else:
    print('Validación desactivada. La evaluación independiente utiliza solo val interna y el checkpoint existente.')

## Interpretación y conclusiones

No hay resultados precalculados. Revisar `result.json`, `worker.log`, las métricas y las
figuras del directorio UUID devuelto. Un estado interrumpido, omitido o early_stopped
no es un entrenamiento completo con las épocas previstas. No comparar fases, splits,
batches o cohortes distintos como una prueba controlada.

Para diagnóstico, la diferencia entre pérdida de train y val no es una medida universal
de sobreajuste: las transformaciones y el muestreo de propuestas pueden ser diferentes.
Las mejoras pequeñas requieren repeticiones futuras con otras semillas y presupuesto
propio; esta sesión con semilla 42 no estima significancia estadística.

Fundamento metodológico: Elgendy, caps. 4, 6 y 7; Lakshmanan et al., caps. 3–8.
Referencias, hipótesis y limitaciones: `EXPERIMENTAL_DESIGN.md`. No se atribuyen a los
libros los hiperparámetros concretos. El puntaje siempre usa el evaluador oficial.